In [ ]:
from io import BytesIO

import joblib
import pandas as pd
from google.colab import files
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

# Upload one cleaned crash CSV.
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one cleaned crash CSV.")

crashes = pd.read_csv(BytesIO(next(iter(uploaded.values()))))
crashes.columns = (
    crashes.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
)

print("Columns in dataset:", list(crashes.columns))

# Create the hour column from times like 17:30:00.
if "hour" not in crashes.columns:
    crashes["hour"] = pd.to_datetime(
        crashes["time"].astype("string").str.strip(),
        format="%H:%M:%S",
        errors="coerce",
    ).dt.hour

print(f"Rows with missing crash hours: {crashes['hour'].isna().sum()}")

# Identify severity column dynamically
severity_col = None
for col in ["accident_severity", "severity", "accident_severity_level", "class"]:
    if col in crashes.columns:
        severity_col = col
        break

if severity_col is None:
    # Fallback to scanning for columns that contain the word 'injury' or 'severity'
    for col in crashes.columns:
        if "injury" in col or "severity" in col:
            severity_col = col
            break

if severity_col:
    print(f"Using '{severity_col}' as the target column. Unique values: {crashes[severity_col].unique()}")
    # Assuming "Slight Injury" or "Slight" is the baseline, other values are high severity.
    crashes["is_high_severity"] = (~crashes[severity_col].astype(str).str.lower().str.contains("slight")).astype(int)
else:
    # If we still can't find it, we will look for an existing 'is_high_severity' column
    if "is_high_severity" not in crashes.columns:
        raise KeyError("Could not automatically identify a severity column. Please make sure the dataset contains a severity column.")

features = [
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "types_of_junction",
    "hour",
]
X = crashes[features]
y = crashes["is_high_severity"].astype(int)

# Hold out 20% of crashes for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Added class_weight='balanced' to handle the heavy class imbalance
model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore"),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Compare with always predicting slight injury (class 0).
model_accuracy = accuracy_score(y_test, predictions)
baseline_accuracy = (y_test == 0).mean()
tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()

print("\nREADME text:")
print(
    f"On the held-out 20% of crashes, the model achieved {model_accuracy:.1%} "
    f"accuracy, compared with {baseline_accuracy:.1%} for always predicting slight injury."
)
print(
    f"It correctly classified {tn} of {tn + fp} slight-injury crashes and "
    f"{tp} of {tp + fn} serious or fatal crashes, missing {fn} serious or fatal crashes."
)

joblib.dump(model, "risk_model.joblib")
files.download("risk_model.joblib")

Saving crashes_clean (3).csv to crashes_clean (3) (5).csv
Columns in dataset: ['time', 'day_of_week', 'age_band_of_driver', 'sex_of_driver', 'educational_level', 'vehicle_driver_relation', 'driving_experience', 'type_of_vehicle', 'owner_of_vehicle', 'service_year_of_vehicle', 'defect_of_vehicle', 'area_accident_occured', 'lanes_or_medians', 'road_allignment', 'types_of_junction', 'road_surface_type', 'road_surface_conditions', 'light_conditions', 'weather_conditions', 'type_of_collision', 'number_of_vehicles_involved', 'number_of_casualties', 'vehicle_movement', 'casualty_class', 'sex_of_casualty', 'age_band_of_casualty', 'casualty_severity', 'work_of_casuality', 'fitness_of_casuality', 'pedestrian_movement', 'cause_of_accident', 'accident_severity', 'accident_hour']
Rows with missing crash hours: 0
Using 'accident_severity' as the target column. Unique values: ['Slight Injury' 'Serious Injury' 'Fatal injury']

README text:
On the held-out 20% of crashes, the model achieved 53.0% accur

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>